# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an example for loading and exploring the [FAIR^2: Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset '{metadata.name}' loaded\n\nDescription: {metadata.description}\n\nKeywords: {metadata.keywords}")
print(f"License: {metadata.license}")


## 2. Data Overview
Review available record sets, fields, and their `@id`s in the Croissant schema.

In [ ]:
# List available record sets with their @id and name
print('Available record sets (by @id):')
for record_set in metadata.record_sets:
    print(f"- @id: {record_set['@id']}, name: {record_set.get('name', '')}")

# Pick the first record set (main protein data table)
main_record_set_id = metadata.record_sets[0]['@id'] if metadata.record_sets else None
if main_record_set_id is not None:
    print(f"\nFields in record set '@id': {main_record_set_id}")
    for field in dataset.metadata.record_set(main_record_set_id).fields:
        print(f"  - @id: {field['@id']} (column: {field.get('column', '')}, name: {field.get('name', '')})")
else:
    print('No record sets found in the dataset metadata.')

## 3. Data Extraction
Load data from the main record set into a DataFrame for analysis. Each record set and field is referenced by its `@id`.

In [ ]:
# Extract the main protein data table into a DataFrame
assert main_record_set_id is not None, "No main record set available!"

record_sets_ids = [rs['@id'] for rs in metadata.record_sets]

dataframes = {}
for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

print(f"Columns available in record set '{main_record_set_id}':")
print(dataframes[main_record_set_id].columns.tolist())

dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
We'll apply some common data processing steps:
- Filter records based on a numeric field (e.g., coverage percentage or peptide counts)
- Normalize a numeric field
- Optionally group by a categorical field (e.g., protein function, if available)

In [ ]:
# Select a representative numeric field for analysis
# We'll look for common protein quant fields: 'Coverage (%)', or 'PeptideCount', or similar. Adjust the field @id below.
# Below, we guess column names based on typical proteomics tables--adjust if your schema uses different names!

df = dataframes[main_record_set_id]
print('First few columns:', df.columns.tolist())

possible_numeric_fields = ['Coverage (%)', 'UniquePeptides', 'Abundance', 'MW (kDa)', 'PeptideCount', 'MolecularWeight', 'TotalIntensity']
numeric_field = None
for col in df.columns:
    if any(name.lower() in col.lower() for name in possible_numeric_fields):
        numeric_field = col
        break

if numeric_field is None:
    # If none matched, pick the first numeric-looking column
    numeric_coltypes = df.select_dtypes(include=['float', 'int']).columns.tolist()
    numeric_field = numeric_coltypes[0] if numeric_coltypes else df.columns[0]
    print(f"Warning: Could not auto-detect numeric field. Using {numeric_field}")

threshold = df[numeric_field].quantile(0.75) if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
filtered_df = df[df[numeric_field] > threshold]
print(f"\nFiltered records with '{numeric_field}' > {threshold:.2f}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df.loc[:, f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized '{numeric_field}' for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a likely categorical field
possible_group_fields = ['Description', 'Gene', 'Protein', 'Family', 'Category', 'Group', 'Function']
group_field = None
for col in df.columns:
    if any(name.lower() in col.lower() for name in possible_group_fields):
        group_field = col
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped filtered data by '{group_field}', showing mean '{numeric_field}':")
    print(grouped_df.head())
else:
    print('(No suitable group field found for grouping. Skipping groupby step.)')

## 5. Visualization
Visualize distributions or relationships in the data. Here, we plot the histogram of the selected numeric field and boxplot by group (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Histogram of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# Boxplot by group field, if present and not too many groups
if group_field and df[group_field].nunique() < 20:
    plt.figure(figsize=(10,4))
    sns.boxplot(data=df, x=group_field, y=numeric_field)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.xticks(rotation=45)
    plt.show()
else:
    print('Too many categories to plot or no group field found.')

## 6. Conclusion

- We successfully loaded the FAIR² protein abundance dataset using `mlcroissant`.
- Main quantitative fields were extracted and explored: we filtered, normalized, and visualized them.
- For detailed downstream biological analysis, further domain-specific steps could leverage the rich metadata encoded in the Croissant schema and the protein annotation fields.  
- For more, see the [dataset description page](https://sen.science/doi/10.71728/senscience.vq4a-28xa).
